# Decision Tree

Model pohon keputusan menghasilkan aturan berbentuk rantai if-else dari akar sampai daun. Data yang dipakai sama dengan latihan KNN, yaitu hasil pemeriksaan sampel tumor.

## Tujuan

- Melatih pohon keputusan dengan hyperparameter tertentu.
- Mengevaluasi kinerja klasifikasi.
- Menggambarkan struktur pohon.
- Mencari hyperparameter terbaik dengan grid search.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

RANDOM_STATE = 123


## Memuat data tumor


In [ ]:
DATASET_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-ML241EN-SkillsNetwork/labs/datasets/tumor.csv"
)
tumor_data = pd.read_csv(DATASET_URL)
tumor_data.head()


In [ ]:
features = tumor_data.iloc[:, :-1]
target = tumor_data.iloc[:, -1].squeeze()
feature_names = features.columns.values

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, stratify=target, random_state=RANDOM_STATE
)


## Pohon keputusan dengan pengaturan bawaan


In [ ]:
default_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
default_tree.fit(X_train, y_train)
predictions = default_tree.predict(X_test)


In [ ]:
def evaluate_metrics(actual, predicted):
    precision, recall, f_beta, _ = precision_recall_fscore_support(
        actual, predicted, average="binary"
    )
    return {
        "accuracy": accuracy_score(actual, predicted),
        "recall": recall,
        "precision": precision,
        "f1score": f_beta,
    }


evaluate_metrics(y_test, predictions)


Hasil awal sudah baik dengan F1 Score sekitar 0.94, tetapi pohonnya sangat dalam dan bercabang banyak.


## Menggambarkan pohon


In [ ]:
def plot_decision_tree(model, names, depth=None):
    figure, axis = plt.subplots(figsize=(20, 12))
    plot_tree(model, feature_names=names, filled=True, max_depth=depth,
              fontsize=8, ax=axis)
    plt.show()


plot_decision_tree(default_tree, feature_names, depth=3)


Warna simpul menandakan kelas mayoritas. Pohon bawaan terlalu besar sehingga aturannya sulit dibaca dan rawan overfitting.


## Menyesuaikan hyperparameter

Kriteria pemisahan diganti menjadi entropy, kedalaman dibatasi sepuluh, dan daun minimal berisi tiga sampel.


In [ ]:
tuned_tree = DecisionTreeClassifier(
    criterion="entropy", max_depth=10, min_samples_leaf=3,
    random_state=RANDOM_STATE
)
tuned_tree.fit(X_train, y_train)
evaluate_metrics(y_test, tuned_tree.predict(X_test))


In [ ]:
plot_decision_tree(tuned_tree, feature_names, depth=3)


## Latihan: pohon dengan kriteria gini

Pohon dilatih dengan `criterion="gini"`, kedalaman maksimum 15, dan daun minimal berisi lima sampel.


In [ ]:
exercise_tree = DecisionTreeClassifier(
    criterion="gini", max_depth=15, min_samples_leaf=5,
    random_state=RANDOM_STATE
)
exercise_tree.fit(X_train, y_train)
evaluate_metrics(y_test, exercise_tree.predict(X_test))


## Mencari hyperparameter terbaik

Grid search menguji kombinasi kriteria, kedalaman, dan ukuran daun minimal dengan validasi silang lima lipatan.


In [ ]:
param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [5, 10, 15, 20],
    "min_samples_leaf": [1, 2, 5],
}

search = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
).fit(X_train, y_train)
search.best_params_


## Latihan: pohon dengan parameter terbaik


In [ ]:
best_tree = DecisionTreeClassifier(**search.best_params_, random_state=RANDOM_STATE)
best_tree.fit(X_train, y_train)
evaluate_metrics(y_test, best_tree.predict(X_test))


In [ ]:
plot_decision_tree(best_tree, feature_names, depth=3)


## Ringkasan

Pembatasan kedalaman dan ukuran daun menurunkan kompleksitas pohon tanpa mengorbankan akurasi. Pohon hasil grid search lebih mudah dibaca dan lebih stabil pada data baru.
